# analysis.host-range

In this analysis, we want to quantify the relationships between bacteria host range and other ranges in the system (virus host range, bacteria range, virus range.)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
import seaborn as sns
from scipy import stats
import numpy as np
import powerlaw
import networkx as nx
from scipy.stats import kruskal, mannwhitneyu
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
si.toc()

In [ ]:
db.toc().df()

## Host range calculations

We compute the host range simply as the number of hosts that a given bacteria can infect.

In [ ]:
metadata = db.conn.sql('SELECT * FROM D_sites').df()
bacteria_hits = db.conn.sql('SELECT * FROM D_PABHits').df()
bacteria_hits = pd.merge(metadata, bacteria_hits, on='library', how='left').dropna(subset='taxid')
bacteria_hits['taxid'] = bacteria_hits['taxid'].astype(int)
bacteria_host_range = bacteria_hits.value_counts(
    ['host_taxon', 'scientific_name']
    ).reset_index().value_counts(
        ['scientific_name']
    ).reset_index().rename(columns={'count': 'host_range'})

bacteria_host_range

In [ ]:
virus_hits = db.conn.sql('SELECT * FROM D_virusHits').df()
virus_hits = pd.merge(metadata, virus_hits, on='library', how='left')
# virus_hits['taxid'] = virus_hits['taxid'].astype(int)
virus_host_range = virus_hits.value_counts(
    ['host_taxon', 'scientific_name']
    ).reset_index().value_counts(
        ['scientific_name']
    ).reset_index().rename(columns={'count': 'host_range'})

virus_host_range

In [ ]:
assert(virus_host_range.describe().loc['count', 'host_range'] == 158)
print("Virus Max Host Range = ", virus_host_range.describe().loc['max', 'host_range'])
print("Virus Mean Host Range = ", virus_host_range.describe().loc['mean', 'host_range'])
print("Virus Min Host Range = ", virus_host_range.describe().loc['min', 'host_range'])

In [ ]:
assert(bacteria_host_range.describe().loc['count', 'host_range'] == 127)
print("PAB Max Host Range = ", bacteria_host_range.describe().loc['max', 'host_range'])
print("PAB Mean Host Range = ", bacteria_host_range.describe().loc['mean', 'host_range'])
print("PAB Min Host Range = ", bacteria_host_range.describe().loc['min', 'host_range'])

In [ ]:
bacteria_host_range['kingdom'] = 'bacteria'
virus_host_range['kingdom'] = 'virus'
host_range = pd.concat([bacteria_host_range, virus_host_range])

db.save_dataframe(host_range, "D_hostRanges", "Bacteria and virus host-ranges")
host_range

In [ ]:

max_range = (host_range['host_range'].max() // 5) + 2
bins = np.arange(0, max_range * 5, 5)
g = sns.displot(data=host_range, x='host_range', col='kingdom', height=2.0, bins=bins)
g.axes[0, 0].set_yscale('log')
g.axes[0, 0].set_xticks([0, 30, 60, 90])
g.set_xlabels("Host range")
g.savefig("figures/displot.host-range.colbykingdom.svg")

## Host range distributions

The results that we observe indicate that a very few virus or bacteria have a remarkable host-range, while most of them have very few hits. This could mean that the host-range might be distributed according to a heavy tailed distribution in organism-host network. We use the `powerlaw` package to compute whether the data here could be fitted to a heavy tailed model better than a simple exponential model.

In [ ]:
heavy_tail_model_lrtest = []

pwl_virus = powerlaw.Fit(data=virus_host_range['host_range'], discrete=True)
pwl_bacteria = powerlaw.Fit(data=bacteria_host_range['host_range'], discrete=True)

R, p = pwl_virus.distribution_compare('power_law', 'exponential')
heavy_tail_model_lrtest.append({'distribution':'virus', 'dist1': 'power-law', 'dist2': 'exponential', 'R':R, 'p-value':p})
R, p = pwl_virus.distribution_compare('lognormal', 'exponential')
heavy_tail_model_lrtest.append({'distribution':'virus', 'dist1': 'lognormal', 'dist2': 'exponential', 'R':R, 'p-value':p})
R, p = pwl_bacteria.distribution_compare('power_law', 'exponential')
heavy_tail_model_lrtest.append({'distribution':'bacteria', 'dist1': 'power-law', 'dist2': 'exponential', 'R':R, 'p-value':p})
R, p = pwl_bacteria.distribution_compare('lognormal', 'exponential')
heavy_tail_model_lrtest.append({'distribution':'bacteria', 'dist1': 'lognormal', 'dist2': 'exponential', 'R':R, 'p-value':p})
heavy_tail_model_lrtest = pd.DataFrame.from_records(heavy_tail_model_lrtest)

db.save_dataframe(
    heavy_tail_model_lrtest, table_name="T_hostRangeslogRatioTest", description="Power-law, Lognormal, and Exponential fit log-ratio test"
)

heavy_tail_model_lrtest

In [ ]:
R, p = pwl_virus.distribution_compare('power_law', 'exponential')
R, p = pwl_bacteria.distribution_compare('lognormal', 'exponential')
heavy_tail_model_fits = pd.DataFrame.from_records([
    {"distribution": "bacteria", "model": "exponential", "log-likelihood": pwl_bacteria.exponential.loglikelihood},
    {"distribution": "bacteria", "model": "lognormal", "log-likelihood": pwl_bacteria.lognormal.loglikelihood},
    {"distribution": "bacteria", "model": "power-law", "log-likelihood": pwl_bacteria.power_law.loglikelihoods(pwl_bacteria.data).sum()},
    {"distribution": "virus", "model": "exponential", "log-likelihood": pwl_virus.exponential.loglikelihood},
    {"distribution": "virus", "model": "lognormal", "log-likelihood": pwl_virus.lognormal.loglikelihood},
    {"distribution": "virus", "model": "power-law", "log-likelihood": pwl_virus.power_law.loglikelihoods(pwl_virus.data).sum()},
])

db.save_dataframe(
    heavy_tail_model_fits, table_name="T_hostRangesFits", description="Power-law, Lognormal, and exponential fits"
)
heavy_tail_model_fits

In [ ]:
TableS8 = pd.merge(
    pd.merge(
        heavy_tail_model_lrtest,
        heavy_tail_model_fits.query('model == "power-law" or model == "lognormal"'), 
        left_on=['distribution', 'dist1'], 
        right_on=['distribution', 'model']
    ),
    heavy_tail_model_fits.query('model == "exponential"'),
    left_on=['distribution', 'dist2'], 
    right_on=['distribution', 'model']
).rename(columns={'distribution': 'kingdom', 'log-likelihood_x': 'dist1_loglikelihood','log-likelihood_y': 'dist2_loglikelihood',})[
    ['kingdom', 'dist1', 'dist2', 'R', 'p-value', 'dist1_loglikelihood', 'dist2_loglikelihood']
]
si.save_dataframe(
    TableS8, table_name="TableS8", description="Host-range long-tail model fits"
)
TableS8

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(6, 3)
pwl_bacteria.plot_ccdf(ax=ax[0], marker='o')
pwl_bacteria.power_law.plot_ccdf(ax=ax[0], color='black', linestyle='-')
pwl_bacteria.exponential.plot_ccdf(ax=ax[0], color='red', linestyle='-')
pwl_bacteria.lognormal.plot_ccdf(ax=ax[0], color='black', linestyle='--')

pwl_virus.plot_ccdf(ax=ax[1], marker='o')
pwl_virus.power_law.plot_ccdf(ax=ax[1], color='black', linestyle='-')
pwl_virus.exponential.plot_ccdf(ax=ax[1], color='red', linestyle='-')
pwl_virus.lognormal.plot_ccdf(ax=ax[1], color='black', linestyle='--')

ax[0].set_title("Bacteria Host range")
ax[1].set_title("Virus Host range ")

ax[0].set_xlabel("Host range")
ax[1].set_xlabel("Host range")

ax[0].set_ylabel("CCDF")
ax[1].set_ylabel("CCDF")

fig.tight_layout()
fig.savefig("figures/ccdfplot.host-range-distributions.svg")

**Results**: It seems that virus are distributed following some sort of heavy-tailed degree distribution, while bacteria are not clearly following such a type of distribution.

## Host Range and Cooccurrence



In [ ]:
cooccurrence_network = nx.read_graphml("output/network.coocurrence.virusbact-bylibrary.trans.graphml")
cooccurrence_network.number_of_edges()

In [ ]:
cooccurrence_network.nodes()

We compute host-ranges

In [ ]:
virus_host_range = virus_hits.drop_duplicates(
    subset=['scientific_name', 'host_taxon']
).value_counts('scientific_name').reset_index()
bacteria_host_range = bacteria_hits.drop_duplicates(
    subset=['scientific_name', 'host_taxon']
).value_counts('scientific_name').reset_index()
virus_host_range['kingdom'] = 'virus'
bacteria_host_range['kingdom'] = 'bacteria'
host_range = pd.concat([virus_host_range, bacteria_host_range])
host_range

In [ ]:
hits = pd.concat([
    bacteria_hits[['host_taxon', 'scientific_name']],
    virus_hits[['host_taxon', 'scientific_name']],
]).drop_duplicates(['host_taxon', 'scientific_name'], keep='first')
hits

In [ ]:

organism_cooccurrences = []
for hit in hits.scientific_name.unique():
    if hit in cooccurrence_network.nodes():
        organism_cooccurrences.append({
            "scientific_name": hit,
            "cooc_net_degree": cooccurrence_network.degree[hit]
        })
    else:
        organism_cooccurrences.append({
            "scientific_name": hit,
            "cooc_net_degree": 0
        })
organism_cooccurrences = pd.DataFrame.from_records(organism_cooccurrences)
organism_cooccurrences['cooc_net_degree'] = organism_cooccurrences['cooc_net_degree'].fillna(0)
organism_cooccurrences['does_cooccur'] = organism_cooccurrences['cooc_net_degree'] > 0
host_range_cooccurrence = pd.merge(organism_cooccurrences, host_range,  on='scientific_name')

host_range_cooccurrence
    

In [ ]:
# host_range_cooccurrence = pd.merge(cooccurrence, host_range, left_on='name', right_on='scientific_name', how='right')
# host_range_cooccurrence['does_cooccur'] = (1-host_range_cooccurrence['name'].isna()).astype(bool).astype(str)
# host_range_cooccurrence['degree'] = host_range_cooccurrence['degree'].fillna(0)
# host_range_cooccurrence



In [ ]:
host_range_cooccurrence = host_range_cooccurrence.rename(columns={'count': 'host_range'})[[
    'scientific_name', 'kingdom', 'cooc_net_degree', 'host_range', 'does_cooccur'
]]

In [ ]:
g = sns.catplot(host_range_cooccurrence, x='does_cooccur', y='host_range',  kind='box', height=2.5, aspect=0.8)
g.set_xlabels("Coccurrence?")
g.set_ylabels("Host-range")

In [ ]:
g = sns.catplot(host_range_cooccurrence, x='does_cooccur', y='host_range',  kind='box', height=2.5, aspect=0.8, col='kingdom')
g.set_xlabels("Coccurrence?")
g.set_ylabels("Host-range")

In [ ]:
host_range_cooccurrence[['does_cooccur', 'kingdom', 'host_range']].groupby(['kingdom', 'does_cooccur']).mean().round(1).reset_index()

In [ ]:
g = sns.lmplot(
    host_range_cooccurrence, x='cooc_net_degree', y='host_range', height=2.0,
    scatter_kws={'alpha':0.5, 's':10, 'color': 'black'}, line_kws={'color': 'red', 'linestyle':'--'}
)
g.set_xlabels("Cooccurrence net. degree")
g.set_ylabels("Host range")
g.savefig("figures/linreg.coocurrence-degree.host-range.svg")

In [ ]:
test = stats.linregress(host_range_cooccurrence['cooc_net_degree'], host_range_cooccurrence['host_range'])

test_results = pd.DataFrame.from_records([
    {"key": "title", "value":"Cooccurrence versus host-range"},
    {"key": "test-type", "value":"Regression"},
    {"key": "H0", "value":""},
    {"key": "H1", "value":""},
    {"key": "p-value", "value": test.pvalue}, #type: ignore
    {"key": "significative", "value": test.pvalue < 0.05}, #type: ignore
    {"key": "intercept", "value": test.intercept}, #type: ignore
    {"key": "slope", "value": test.slope}, #type: ignore
    {"key": "r-value", "value": test.rvalue}, #type: ignore
    {"key": "R2", "value": test.rvalue ** 2} #type: ignore

])

db.save_dataframe(
    test_results, table_name="T_coocHostRangeCorr", 
    description="Correlation test between number of cooccurrences and the host-range"
)
test_results


In [ ]:
g = sns.lmplot(
    host_range_cooccurrence, x='cooc_net_degree', y='host_range', height=2.0, col='kingdom',
    scatter_kws={'alpha':0.5, 's':10, 'color': 'black'}, line_kws={'color': 'red', 'linestyle':'--'}
)
g.set_xlabels("Cooccurrence degree")
g.set_ylabels("Host range")
g.savefig("figures/linreg.coocurrence-degree.host-range.colbykingdom.svg")

In [ ]:
test = stats.linregress(
    host_range_cooccurrence.query('kingdom == "virus"')['cooc_net_degree'], 
    host_range_cooccurrence.query('kingdom == "virus"')['host_range'])

test_results = pd.DataFrame.from_records([
    {"key": "title", "value":"Cooccurrence versus host-range in virus"},
    {"key": "test-type", "value":"Regression"},
    {"key": "H0", "value":""},
    {"key": "H1", "value":""},
    {"key": "p-value", "value": test.pvalue}, #type: ignore
    {"key": "significative", "value": test.pvalue < 0.05}, #type: ignore
    {"key": "intercept", "value": test.intercept}, #type: ignore
    {"key": "slope", "value": test.slope}, #type: ignore
    {"key": "r-value", "value": test.rvalue}, #type: ignore
    {"key": "R2", "value": test.rvalue ** 2} #type: ignore

])

db.save_dataframe(
    test_results, table_name="T_coocHostRangeCorrVir", 
    description="Correlation test between number of cooccurrences and the host-range in Virus"
)
test_results


In [ ]:
test = stats.linregress(
    host_range_cooccurrence.query('kingdom == "bacteria"')['cooc_net_degree'], 
    host_range_cooccurrence.query('kingdom == "bacteria"')['host_range'])

test_results = pd.DataFrame.from_records([
    {"key": "title", "value":"Cooccurrence versus host-range in Bacteria"},
    {"key": "test-type", "value":"Regression"},
    {"key": "H0", "value":""},
    {"key": "H1", "value":""},
    {"key": "p-value", "value": test.pvalue}, #type: ignore
    {"key": "significative", "value": test.pvalue < 0.05}, #type: ignore
    {"key": "intercept", "value": test.intercept}, #type: ignore
    {"key": "slope", "value": test.slope}, #type: ignore
    {"key": "r-value", "value": test.rvalue}, #type: ignore
    {"key": "R2", "value": test.rvalue ** 2} #type: ignore

])

db.save_dataframe(
    test_results, table_name="T_coocHostRangeCorrBact", 
    description="Correlation test between number of cooccurrences and the host-range in Bacteria"
)
test_results


In [ ]:
test_results = []
test = mannwhitneyu(
    host_range_cooccurrence.query('does_cooccur == True')['host_range'],
    host_range_cooccurrence.query('does_cooccur == False')['host_range'],
)

test_results.append(
    {
        "organisms": "all", "statistic": test.statistic, "p-value": test.pvalue
    }
)
test = mannwhitneyu(
    host_range_cooccurrence.query('does_cooccur == True').query('kingdom == "bacteria"')['host_range'],
    host_range_cooccurrence.query('does_cooccur == False').query('kingdom == "bacteria"')['host_range'],
)
test_results.append(
    {
        "organisms": "bacteria", "statistic": test.statistic, "p-value": test.pvalue
    }
)
mannwhitneyu(
    host_range_cooccurrence.query('does_cooccur == True').query('kingdom == "virus"')['host_range'],
    host_range_cooccurrence.query('does_cooccur == False').query('kingdom == "virus"')['host_range'],
)
test_results.append(
    {
        "organisms": "virus", "statistic": test.statistic, "p-value": test.pvalue
    }
)
test_results = pd.DataFrame.from_records(test_results)

db.save_dataframe(
    test_results, table_name="T_coocHostRangeMW", 
    description="Multiple Mann-Whitney tests on the Host-range of cooccurring organisms versus not cooccurring organisms"
)
test_results

In [ ]:
si.save_dataframe(
    host_range_cooccurrence, 'TableS9',
    'Host range and cooccurrence network degree'
)
host_range_cooccurrence

## Host-range by habitats

In this block, we want to check whether hosts exhibit different host-ranges across different habitats.

We will:

1. Compute the per-habitat host range from the data.
2. Generate vectors containing that data
3. Run an ANOVA test
4. Save the result

### Step 1: Compute per-habitat host-range

We simply add the habitat inside the host-range.

In [ ]:

# bacteria_host_range_byhabitat = bacteria_hits.value_counts(
#     ['host_taxon', 'habitat', 'scientific_name']
#     ).reset_index().value_counts(
#         ['scientific_name', 'habitat']
#     ).reset_index().rename(columns={'count': 'host_range'})

# virus_host_range_byhabitat = virus_hits.value_counts(
#     ['host_taxon', 'habitat', 'scientific_name']
#     ).reset_index().value_counts(
#         ['scientific_name', 'habitat']
#     ).reset_index().rename(columns={'count': 'host_range'})

# bacteria_host_range_byhabitat['kingdom'] = 'bacteria'
# virus_host_range_byhabitat['kingdom'] = 'virus'
# host_range_byhabitat = pd.concat(
#     [
#         bacteria_host_range_byhabitat, 
#         virus_host_range_byhabitat
#     ]
# )

# # db.save_dataframe(host_range, "Host_ranges", "Bacteria and virus host-ranges")
# host_range_byhabitat


In [ ]:
virus_host_range_byhabitat_pvt = db.conn.sql('SELECT * FROM D_virusOTUs').df()[['scientific_name', 'Crop_HR', 'Edge_HR', 'Oak_HR', 'Wasteland_HR']]
virus_host_range_byhabitat_pvt['kingdom'] = 'virus'

bact_host_range_byhabitat_pvt = db.conn.sql('SELECT * FROM D_PABOTUs').df()[['scientific_name', 'Crop_HR', 'Edge_HR', 'Oak_HR', 'Wasteland_HR']]
bact_host_range_byhabitat_pvt['kingdom'] = 'bacteria'

host_range_byhabitat_pvt = pd.concat([virus_host_range_byhabitat_pvt, bact_host_range_byhabitat_pvt])
host_range_byhabitat_pvt



An issue that could affect the calculations would be the absence of zeros in those organisms that were not detected in a given habitat. Therefore, we neeed to take them into account. We can do this by pivoting our datable and filling with zeros.

In [ ]:
host_range_byhabitat_melt = host_range_byhabitat_pvt.melt(id_vars=['scientific_name', 'kingdom'], value_vars=['Crop_HR', 'Edge_HR', 'Oak_HR', 'Wasteland_HR'])
host_range_byhabitat_melt['variable'] = host_range_byhabitat_melt['variable'].apply(lambda x: x.replace('_HR', ''))
host_range_byhabitat_melt = host_range_byhabitat_melt.rename(columns={'variable':'habitat', 'value':'host_range'}).query('host_range != 0.0').copy()
host_range_byhabitat_melt

Now, the vectors are simply the columns of the pivotted table.

In [ ]:
g = sns.catplot(
    host_range_byhabitat_melt,
    x='habitat', y='host_range', hue='habitat', height=2.0, aspect=1.5,
    palette=conf['habitat_palette'], kind='box', whis=100000000000.0
)
g.set_ylabels("per-habitat Host range")
# g.ax.set_yscale("log")

In [ ]:
g = sns.catplot(
    host_range_byhabitat_melt,
    x='habitat', y='host_range', hue='habitat', height=2.0, aspect=1.5, col='kingdom',
    palette=conf['habitat_palette'], kind='box', whis=100000000000.0
)
g.set_ylabels("per-habitat Host range")

# g.ax.set_yscale("log")

In [ ]:
test_1 = kruskal(
    host_range_byhabitat_melt.query('habitat == "Crop"')['host_range'].values,
    host_range_byhabitat_melt.query('habitat == "Edge"')['host_range'].values,
    host_range_byhabitat_melt.query('habitat == "Wasteland"')['host_range'].values,
    host_range_byhabitat_melt.query('habitat == "Oak"')['host_range'].values,
)
test_1

In [ ]:
test_2 = kruskal(
    host_range_byhabitat_melt.query('kingdom == "bacteria"').query('habitat == "Crop"')['host_range'].values,
    host_range_byhabitat_melt.query('kingdom == "bacteria"').query('habitat == "Edge"')['host_range'].values,
    host_range_byhabitat_melt.query('kingdom == "bacteria"').query('habitat == "Wasteland"')['host_range'].values,
    host_range_byhabitat_melt.query('kingdom == "bacteria"').query('habitat == "Oak"')['host_range'].values,
)
test_2

In [ ]:
test_3 = kruskal(
    host_range_byhabitat_melt.query('kingdom == "virus"').query('habitat == "Crop"')['host_range'].values,
    host_range_byhabitat_melt.query('kingdom == "virus"').query('habitat == "Edge"')['host_range'].values,
    host_range_byhabitat_melt.query('kingdom == "virus"').query('habitat == "Wasteland"')['host_range'].values,
    host_range_byhabitat_melt.query('kingdom == "virus"').query('habitat == "Oak"')['host_range'].values,
)
test_3

In [ ]:
results = pd.DataFrame.from_records(
    [
        {
            "test": "KW", 
            "data": "All",
            "H": test_1.statistic,
            "p-value": test_1.pvalue
        },
        {
            "test": "KW", 
            "data": "Bacteria",
            "H": test_2.statistic,
            "p-value": test_2.pvalue
        },
        {
            "test": "KW", 
            "data": "Virus",
            "H": test_3.statistic,
            "p-value": test_3.pvalue
        }
    ]
)
results

In [ ]:
db.save_dataframe(host_range_byhabitat_melt, "d_HostRangebyHab", "Bacteria and virus host-ranges by habitat")

### Post-hoc host-range by habitat

In [ ]:
post_hoc_hostrange_by_habitat = []
for org in ['virus', 'bacteria', 'all']:
    if org != 'all':
        df = host_range_byhabitat_melt.query('kingdom == "{0}"'.format(org))
    else:
        df = host_range_byhabitat_melt.copy()
    for h1 in ['Crop', 'Edge', 'Wasteland', 'Oak']:
        for h2 in ['Crop', 'Edge', 'Wasteland', 'Oak']:
            if h1 != h2:
                kw_h, pval = stats.mannwhitneyu(
                    df.query('habitat == "{0}"'.format(h1))['host_range'].values,
                    df.query('habitat == "{0}"'.format(h2))['host_range'].values,
                    alternative='less'
                )
                significative = pval < 0.05
                post_hoc_hostrange_by_habitat.append(
                    {'kingdom': org, 'group_1': h1, 'group_2': h2, 'U': kw_h, 'p-val': pval, 'sign': significative}
            )
post_hoc_hostrange_by_habitat = pd.DataFrame.from_records(post_hoc_hostrange_by_habitat)

db.save_dataframe(
    df=post_hoc_hostrange_by_habitat, table_name="T_hostRangeByHabitat",
    description="Post-Hoc Mann Whitney U analysis on host range across different habitats and kingdoms"
)
si.save_dataframe(
    df=post_hoc_hostrange_by_habitat, table_name="TableS11",
    description="Mann-Whitney U post-hoc test on site-diversity by habitat"
)
post_hoc_hostrange_by_habitat

In [ ]:
db.conn.close()
si.conn.close()